# 🚀 FinanceBench SEC Filings Ingestion Pipeline (Kaggle Cloud)

This notebook offloads the batch ingestion of **368 SEC Filings (10-Ks & 10-Qs)** to Kaggle's high-memory cloud infrastructure:
- **ChromaDB:** Generates dense vector embeddings using `sentence-transformers/all-MiniLM-L6-v2`.
- **BM25 Index:** Constructs an exact-match keyword BM25 sparse index.
- **Neo4j AuraDB (Cloud):** Ingests financial statements, XBRL-aligned line items, and parent-child footnotes directly into your **Neo4j AuraDB Free** instance over encrypted Bolt (`neo4j+s://`).

---
### 📋 Prerequisites:
1. **Neo4j AuraDB Free Account:** Create a free database at [neo4j.com/cloud/platform/aura-graph-database/](https://neo4j.com/cloud/platform/aura-graph-database/). Note your URI (`neo4j+s://...`) and password.
2. **Kaggle Dataset:** Upload `financebench_dataset.zip` (containing `pdfs/` and metadata `.jsonl` files) as a private Kaggle dataset.
3. **Kaggle Notebook Settings:**
   - **Accelerator:** GPU T4 x2 (recommended) or CPU.
   - **Internet:** **On** (required to connect to Neo4j AuraDB and download packages).


In [ ]:
# 1. Install Required Dependencies (compatible with Kaggle base environment)
!pip install -q --no-warn-conflicts \
    "opentelemetry-api<1.39.0,>=1.36.0" \
    "opentelemetry-sdk<1.39.0,>=1.36.0" \
    "opentelemetry-proto==1.38.0" \
    "opentelemetry-exporter-otlp-proto-common==1.38.0" \
    chromadb \
    rank-bm25 \
    pypdf \
    neo4j \
    langchain-text-splitters \
    langchain-openai \
    sentence-transformers \
    flashrank \
    pydantic || true

# Verify imports
import chromadb
import neo4j
import pypdf
print("✅ Core packages imported successfully!")

## ⚙️ 2. Configure Credentials & Paths
Enter your **Neo4j AuraDB** credentials below. You can either enter them directly or use [Kaggle Secrets](https://www.kaggle.com/product-features/secrets).

In [ ]:
import os
import sys
from pathlib import Path

# --- NEO4J AURADB CONFIGURATION ---
# Verified working credentials for instance 3c576ecf
NEO4J_URI = os.getenv("NEO4J_URI", "neo4j+ssc://3c576ecf.databases.neo4j.io")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "3c576ecf")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "pFKPo3ZU0CNG8DD9h6iGaIInBJ2Nc_MnzUYhFCbFeK8")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "3c576ecf")

# Set in environment for pipeline detection
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE
os.environ["NEO4J_TIMEOUT"] = "20.0"

# --- PATH CONFIGURATION ---
# Kaggle Input Dataset location (dynamically searches wherever dataset was mounted/extracted)
INPUT_BASE = Path("/kaggle/input")

# 1. Find PDF Directory
pdf_dirs = list(INPUT_BASE.glob("**/pdfs"))
if pdf_dirs:
    PDF_DIR = pdf_dirs[0]
else:
    pdf_files = list(INPUT_BASE.glob("**/*.pdf"))
    PDF_DIR = pdf_files[0].parent if pdf_files else Path("/kaggle/input/financebench-dataset/pdfs")

# 2. Find Metadata JSONL File
meta_files = list(INPUT_BASE.glob("**/financebench_document_information.jsonl"))
METADATA_PATH = meta_files[0] if meta_files else PDF_DIR.parent / "financebench_document_information.jsonl"

# Working output directory (persists across notebook save)
OUTPUT_DIR = Path("/kaggle/working/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHROMA_PERSIST_DIR = OUTPUT_DIR / "chroma_db"
BM25_PATH = OUTPUT_DIR / "bm25_financebench.json"
MANIFEST_PATH = OUTPUT_DIR / "financebench_manifest.json"

print(f"📁 PDF Directory:    {PDF_DIR} (Files: {len(list(PDF_DIR.glob('*.pdf')) if PDF_DIR.exists() else 0)})")
print(f"📄 Metadata Path:    {METADATA_PATH} (Exists: {METADATA_PATH.exists()})")
print(f"💾 ChromaDB Path:    {CHROMA_PERSIST_DIR}")
print(f"⚡ Neo4j URI:        {NEO4J_URI} (User: {NEO4J_USERNAME})")

## 🔌 3. Verify Neo4j AuraDB Connection
Test connectivity to make sure the encrypted TLS Bolt handshake succeeds before running the full ingestion.

In [ ]:
import neo4j

print(f"Testing connection to Neo4j AuraDB as user '{NEO4J_USERNAME}'...")
try:
    driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run("RETURN 'Connection Successful!' AS status, datetime() AS time")
        record = result.single()
        print(f"✅ {record['status']} (Server time: {record['time']})")
    driver.close()
except Exception as e:
    print(f"❌ Could not connect to Neo4j AuraDB: {e}")

## 📦 4. Configure Python Import Path
Finds and adds `src/` to the Python sys.path whether Kaggle auto-extracted the zip into a folder or kept it as an archive.

In [ ]:
# 1. Check if Kaggle automatically unzipped the code into a folder containing 'src'
src_candidates = [p for p in INPUT_BASE.glob("**/src") if p.is_dir()]

if src_candidates:
    src_path = str(src_candidates[0])
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    print(f"✅ Discovered and added to sys.path: {src_path}")
else:
    # 2. Check if financial_intelligence_code.zip is present
    code_zips = list(INPUT_BASE.glob("**/financial_intelligence_code.zip"))
    if code_zips:
        print(f"Extracting {code_zips[0]} to /kaggle/working/...")
        !unzip -q -o {code_zips[0]} -d /kaggle/working/
        sys.path.insert(0, "/kaggle/working/src")
        print("✅ Unzipped and added /kaggle/working/src to Python path.")
    else:
        print("❌ ERROR: Neither 'src' directory nor 'financial_intelligence_code.zip' found in /kaggle/input!")
        print("   Current /kaggle/input structure:")
        !ls -R /kaggle/input

## ⚙️ 5. Run Full 368-Filing Batch Ingestion
Runs `FinanceBenchIngestionPipeline` across all 368 filings:
1. Parses 2D tables into Markdown matrices and identifies canonical US-GAAP financial line items.
2. Resolves footnote references to their disclosure texts.
3. Ingests line items and relationships into **Neo4j AuraDB**.
4. Upserts parent-child chunks into **ChromaDB** and updates **BM25** index.
5. Updates `financebench_manifest.json` for idempotency.

In [ ]:
from langchain_rag.finance_pipeline import FinanceBenchIngestionPipeline

print("=" * 70)
print("   LAUNCHING FINANCEBENCH BATCH INGESTION (CHROMA + BM25 + NEO4J)")
print("=" * 70)

pipeline = FinanceBenchIngestionPipeline(
    pdf_dir=str(PDF_DIR),
    metadata_path=str(METADATA_PATH),
    persist_directory=str(CHROMA_PERSIST_DIR),
    collection_name="financebench",
    manifest_path=str(MANIFEST_PATH),
    bm25_path=str(BM25_PATH),
    max_chunk_chars=1200,
    batch_size=50,
    enable_bm25=True,
    enable_graph=True,
)

# Progress display callback
def on_progress(info: dict) -> None:
    curr = info["current"]
    total = info["total"]
    pct = (curr / total) * 100.0 if total else 0.0
    if curr % 5 == 0 or curr == total:
        print(f"[{curr:>3}/{total:>3}] ({pct:5.1f}%) | {info['file'][:32]:<32} (+{info['chunks']} chunks)")

# Ingest all documents
stats = pipeline.ingest(
    force=False,
    dry_run=False,
    sample=None,  # None = process all 368 filings
    max_pages_per_doc=None,  # None = all pages
    progress_callback=on_progress,
)

print("\n" + "=" * 70)
print("   INGESTION COMPLETED")
print("=" * 70)
print(f"Total Files Discovered  : {stats['total_files_discovered']}")
print(f"Files Indexed           : {stats['files_to_index']}")
print(f"Chunks Upserted         : {stats['chunks_upserted']}")
print(f"Graph Entities Ingested : {stats.get('graph_entities_ingested', 0)}")
print(f"Execution Duration      : {stats['duration_seconds']:.2f}s")
print("=" * 70)

## 🔍 6. Verify Knowledge Graph & Vector Store
Run a sample verification query against both Neo4j AuraDB and ChromaDB to verify data integrity.

In [ ]:
# 1. Verify Neo4j AuraDB counts
driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
with driver.session() as session:
    companies = session.run("MATCH (c:Company) RETURN count(c) AS count").single()["count"]
    filings = session.run("MATCH (f:Filing) RETURN count(f) AS count").single()["count"]
    line_items = session.run("MATCH (l:LineItem) RETURN count(l) AS count").single()["count"]
    footnotes = session.run("MATCH (fn:Footnote) RETURN count(fn) AS count").single()["count"]
    
    print(f"📊 Neo4j AuraDB Statistics:")
    print(f"   - Companies:   {companies}")
    print(f"   - Filings:     {filings}")
    print(f"   - Line Items:  {line_items}")
    print(f"   - Footnotes:   {footnotes}")
driver.close()

# 2. Verify ChromaDB & Hybrid Search
test_query = "What is Apple's revenue and R&D spending in 2023?"
results = pipeline.query(query_text=test_query, top_k=3, search_type="hybrid", rerank=True)
print(f"\n🔎 Test Retrieval for: '{test_query}'")
for idx, r in enumerate(results, start=1):
    meta = r.get("metadata", {})
    print(f"   [{idx}] {meta.get('company')} | {meta.get('doc_name')} | Page {meta.get('page')}")

## 💾 7. Package Data Bundle for Download
Creates a single compressed archive `/kaggle/working/financial_rag_data_bundle.zip` containing:
- `data/chroma_db/` (all vector embeddings)
- `data/bm25_financebench.json` (keyword search index)
- `data/financebench_manifest.json` (idempotency hashes)

In [ ]:
!zip -q -r /kaggle/working/financial_rag_data_bundle.zip /kaggle/working/data

import os
bundle_size = os.path.getsize("/kaggle/working/financial_rag_data_bundle.zip") / (1024 * 1024)
print(f"✅ Successfully created /kaggle/working/financial_rag_data_bundle.zip ({bundle_size:.1f} MB)")
print("📥 You can now download this file directly from the Kaggle Notebook 'Output' pane!")

## 🏁 8. Next Steps on Your Local Machine

1. **Download:** In the Kaggle notebook sidebar, navigate to the **Output** tab and download `financial_rag_data_bundle.zip`.
2. **Extract:** Unzip it into your local repository's `data/` directory (`c:\Users\Asus\Langchain\data\`).
3. **Update `.env`:** Add your Neo4j AuraDB credentials:
   ```env
   NEO4J_URI=neo4j+s://<your-id>.databases.neo4j.io
   NEO4J_USERNAME=neo4j
   NEO4J_PASSWORD=<your-password>
   ```
4. **Query & Evaluate:** Your local LangGraph agent and FastAPI endpoint can now instantly query all 368 companies with zero local ingestion overhead!